## LLM as a Judge

In [2]:
# Earlier we generated RAG answers for our ground truth questions. Now we need to decide whether these answers are good enough.

# For offline evaluation, we have three things:

# -- the original FAQ answer
# -- the question generated from that answer
# -- the answer generated by our RAG pipeline

# An LLM judge is another LLM call that compares these three pieces. 
# We ask it whether the RAG answer recovers the same information as the original answer.

# It can also explain why an answer is bad.

# For example:

# -- the retrieved document might be wrong
# -- the answer might miss the key point
# -- the model might say that it doesn't know

# This approach is useful when exact text matching is too strict. The RAG answer doesn't need to copy the FAQ answer word for word. 
# It needs to answer the question with the same key information.

# This evaluates the full RAG flow in one pass:

# -- search: did we retrieve context that contains the answer?
# -- prompt: did we give the model enough context to answer?
# -- LLM: did the model produce a useful answer from that context?

# If the judge marks an answer as bad, we still need to look at the example. The judge tells us where to investigate. 
# It doesn't replace reading the failing cases.

# Loading the RAG answers
# Start from the CSV we created in the previous lesson:

import pandas as pd

df_answers = pd.read_csv("rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

# Each row has the generated question, the original FAQ answer, and the answer produced by the RAG pipeline.

In [3]:
# A->Q->A' evaluation
# We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers 
# that match the ground truth.

# First, define the output format:

from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )



In [4]:
# The judge returns two fields. The score gives us a metric we can aggregate. The reasoning explains the score, 
# which helps when we look at bad examples.

# First, write the judge instructions. This tells the judge what to compare and how to assign the score.

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
# Then define the prompt template. This is the data we pass to the judge for each answer.

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
# Import the structured-output helper:

from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

# Take one record:

rec = answers[0]

In [7]:
rec

{'question': 'Can I still join the course if I just found it now?',
 'answer_llm': 'Yes — you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [8]:
# Create the judge prompt:

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

prompt

'Question:\nCan I still join the course if I just found it now?\n\nOriginal Answer (ground truth):\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nAI Answer:\nYes — you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [9]:
# Call the judge:

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the ground truth meaning: joining is still possible, and certificate eligibility depends on submitting the project while submissions are open. No key information is missing or incorrect.', score='good')

In [10]:
# Check the cost:

calc_price(usage)

{'input_cost': 0.000219, 'output_cost': 0.0002295, 'total_cost': 0.0004485}

In [11]:
# Now put the same logic into a function:

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage


# Test it on the same record:

eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [12]:
# Running the judge
# Run the evaluation on all answers:

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

# Use the same parallel processing helper:

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)


    

Task was destroyed but it is pending!
task: <Task pending name='Task-151' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\tmp\agentic-RAG-vector-search-orchestration-evaluation-monitoring\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-152' coro=<Kernel.shell_main() running at C:\tmp\agentic-RAG-vector-search-orchestration-evaluation-monitoring\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\tmp\agentic-RAG-vector-search-orchestration-evaluation-monitoring\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\d_local\AppData\Roaming\uv\python\cpython-3.13.9-windows-x86_64-none\Lib\multiprocessing\synchronize.py:22: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  from . import util
Task was destroyed but it is pending!
task: <Task pending name='Task-152' coro=<Kernel.shell_main() running at C:\tmp\agentic-RAG-vector-se

  0%|          | 0/565 [00:00<?, ?it/s]

In [13]:
# Split the results:

evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

# Create a dataframe:

df_eval = pd.DataFrame(evaluations)

# Calculate the total cost:

calc_total_price(usages)

0.39571575000000003

In [15]:
# Check the results:

good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

# Look at the "bad" cases to understand what went wrong:

df_eval[df_eval["score"] == "bad"].head()

Good: 545/565 = 96.46%


,question,document,score,reasoning
1,Is it okay to start late and catch up with the...,74eb249bbf,bad,The AI answer correctly says it is okay to sta...
3,What do I need to do to be eligible for the ce...,74eb249bbf,bad,The ground truth says late joiners are eligibl...
41,Do you know what season or year the course run...,bd31146b0e,bad,The ground truth gives a specific time: Summer...
43,When can I take this course again if I miss it...,bd31146b0e,bad,The ground truth gives a specific retake time:...
46,Where should I start if I want to watch the le...,31456f4b5f,bad,The ground truth says the main entry point sho...


In [16]:
# These rows are often the most useful part of the evaluation. They can show that search retrieved the wrong document. 
# They can also show that the answer is too generic. Sometimes the RAG pipeline says that it doesn't know even though the FAQ had the answer.

# Evaluating the judge
# The judge can be wrong. It may rate an answer as good even though search failed to retrieve the right document. 
# In that case the judge is too lenient. Make the instructions stricter and re-run the evaluation.

# To evaluate the judge, you need to look at the results yourself. Sample some good and bad cases, read the judge reasoning, 
# and check whether you agree with the verdict. You cannot use another judge to evaluate the judge. This is manual work, but it is necessary.

# A practical approach is to build a simple application with Streamlit. Show each question, the original answer, 
# the generated answer, and the judge verdict side by side. Then mark each verdict as correct or incorrect and use that 
# feedback to adjust the judge instructions. This is a lot of trial and error, but it makes the evaluation framework more reliable.

# Saving the results
# Save the judged answers:

df_eval.to_csv("rag-evaluations-new.csv", index=False)